# Incident Response Runbook: Lazarus Group — Solana Developer Machine Compromise

**Tactic:** Initial Access → Credential Access → Exfiltration
**Technique:** T1566.002 (Spearphishing Link) + T1552.001 (Credentials in Files) + T1059.006 (Python)
**Severity:** CRITICAL

## Overview

This runbook covers the Lazarus Group campaign targeting Solana developers (Operation 99,
Operation Graphalgo — 2025). Attackers posed as LinkedIn recruiters or technical interviewers,
lured developers into running malicious npm or PyPI packages, and deployed malware that searched
for the Solana CLI default keypair at `~/.config/solana/id.json`. The private key was
exfiltrated to attacker-controlled infrastructure and used to drain developer and protocol wallets.

## MITRE ATT&CK Mapping

| Technique | ID | Description |
|---|---|---|
| Spearphishing Link | **T1566.002** | Fake LinkedIn recruiter / coding challenge lure |
| Credentials in Files | **T1552.001** | Malware reads `~/.config/solana/id.json` |
| Command and Scripting: Python | **T1059.006** | Malicious PyPI/npm package runs key-exfil script |
| Exfiltration Over C2 Channel | **T1041** | Private key exfiltrated via HTTPS to attacker server |
| Supply Chain Compromise | **T1195.001** | Malicious packages hosted on npm/PyPI |

## Lateral Movement Analysis

Lazarus exploits the trust developers place in package managers and professional networks:

1. **LinkedIn social engineering** — Fake recruiter contacts developer with a "coding challenge" or "algorithm review" repo
2. **Malicious package execution** — Developer runs `pip install graphalgo` or `npm install` in a cloned repo; install script immediately runs malware
3. **Filesystem traversal** — Malware searches for `~/.config/solana/id.json`, `.env` files, browser wallets, and SSH keys
4. **Key exfiltration** — Found key material HTTP POST'd to attacker C2 over HTTPS (blends with normal traffic)
5. **Pivot opportunity** — id.json often funds multiple programs and test wallets; GitHub tokens in `.gitconfig` can further pivot to source code and CI secrets

**Full lateral movement chain:**
`LinkedIn DM → malicious package install → developer machine filesystem → id.json + env files → C2 exfiltration → wallet drain + potential CI/CD pivot`

## Incident Response Phases

1. **Detection & Analysis**
2. **Containment**
3. **Eradication**
4. **Recovery**
5. **Post-Incident Activities**


## Phase 1: Detection & Analysis

### Objectives
- Identify the malicious package and C2 endpoint
- Determine what key material was accessed and exfiltrated
- Assess whether attacker pivoted beyond the developer machine
- Scope all systems that ran the malicious package


In [ ]:
import json
import re
from datetime import datetime
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..', '..')))

from splunk.splunk_data_collector import SplunkDataCollector
from crowdstrike.crowdstrike_response import CrowdStrikeResponse
from iris.iris_integration import IRISIntegration
from misp.misp_integration import MISPIntegration
from shuffle.shuffle_integration import ShuffleIntegration

splunk = SplunkDataCollector()
crowdstrike = CrowdStrikeResponse()
iris = IRISIntegration()
misp = MISPIntegration()
shuffle = ShuffleIntegration()

print("=" * 60)
print("STEP 1: Detection & Analysis — Lazarus id.json Developer Compromise")
print("=" * 60)

detection_time = datetime.now().isoformat()
affected_systems = []
splunk_indicators = []
unique_users = set()
source_hosts = set()

# Detect reads of id.json from unexpected processes
print("\n[QUERY] Detecting unexpected reads of ~/.config/solana/id.json...")
splunk_query = '''
index=endpoint OR index=sysmon
(EventCode=4663 OR event_type="file_read" OR event_type="open_file")
(target_file="*/.config/solana/id.json" OR target_file="*solana*id.json*")
NOT (process_name="solana" OR process_name="anchor" OR process_name="cargo")
| stats count by host, process_name, process_path, user, target_file, _time
| sort -count
'''
try:
    splunk_results = splunk.search_events(splunk_query, timeframe="-72h")
    print(f"   Found {len(splunk_results)} anomalous id.json read events")
except Exception as e:
    print(f"   Splunk query failed: {e}")
    splunk_results = []

for event in splunk_results:
    affected_systems.append({
        'hostname': event.get('host', 'unknown'),
        'process': event.get('process_name', 'unknown'),
        'user': event.get('user', 'unknown'),
        'last_seen': event.get('_time', detection_time)
    })
    unique_users.add(event.get('user', 'unknown'))
    source_hosts.add(event.get('host', 'unknown'))
    splunk_indicators.append({
        'type': 'id_json_read',
        'value': f"host={event.get('host')} process={event.get('process_name')} user={event.get('user')}",
        'context': 'Unexpected process read ~/.config/solana/id.json'
    })

# Detect outbound HTTP from npm/pip install or Python script contexts
print("\n[QUERY] Detecting outbound HTTP from package install processes...")
exfil_query = '''
index=network OR index=proxy
(src_process="python*" OR src_process="node" OR src_process="npm" OR src_process="pip")
http_method=POST
NOT (dest_host="*.pypi.org" OR dest_host="*.npmjs.com" OR dest_host="*.github.com")
| stats count, sum(bytes_out) as total_bytes by src_host, src_process, dest_host, dest_ip, _time
| where total_bytes > 100
| sort -total_bytes
'''
try:
    exfil_results = splunk.search_events(exfil_query, timeframe="-72h")
    print(f"   Found {len(exfil_results)} suspicious outbound POST events from package processes")
    for r in exfil_results:
        splunk_indicators.append({
            'type': 'key_exfiltration',
            'value': f"src={r.get('src_host')} process={r.get('src_process')} dest={r.get('dest_host')} bytes={r.get('total_bytes')}",
            'context': 'Potential key exfiltration from package install/execution context'
        })
        source_hosts.add(r.get('src_host', 'unknown'))
except Exception as e:
    print(f"   Exfiltration query failed: {e}")

# Detect malicious npm/PyPI packages
print("\n[QUERY] Identifying malicious packages from install logs...")
pkg_query = '''
index=ci_logs OR index=endpoint
(sourcetype=npm_install OR sourcetype=pip_install)
(package="graphalgo" OR package="solana-py" OR package="solana-core" OR package="solana-wallet-tools")
| stats count by host, package, package_version, install_user, _time
'''
try:
    pkg_results = splunk.search_events(pkg_query, timeframe="-168h")
    print(f"   Found {len(pkg_results)} installs of suspected malicious packages")
    for r in pkg_results:
        splunk_indicators.append({
            'type': 'malicious_package_install',
            'value': f"package={r.get('package')}@{r.get('package_version')} on {r.get('host')} by {r.get('install_user')}",
            'context': 'Known Lazarus-associated malicious npm/PyPI package installed'
        })
except Exception as e:
    print(f"   Package install log query failed: {e}")

# MISP enrichment for Lazarus C2
print("\n[ENRICHMENT] Checking MISP for Lazarus Group C2 infrastructure...")
misp_results = []
lazarus_c2_iocs = list(set([i.get('value','').split('dest=')[-1].split(' ')[0] for i in splunk_indicators if 'key_exfiltration' in i.get('type','')]))
try:
    for ioc in lazarus_c2_iocs[:5]:
        hits = misp.search_iocs(ioc)
        if hits:
            misp_results.extend(hits)
            print(f"   MISP hit for {ioc}: {len(hits)} events (Lazarus attribution)")
except Exception as e:
    print(f"   MISP enrichment failed: {e}")

print("\n[CASE] Creating IRIS incident case...")
try:
    incident_id = iris.create_case({
        'title': f'Lazarus Group Developer Machine Compromise — id.json exfiltration — {len(affected_systems)} hosts',
        'severity': 'CRITICAL',
        'technique': 'T1566.002 + T1552.001 + T1059.006',
        'threat_actor': 'Lazarus Group (DPRK)',
        'indicators': splunk_indicators
    })
    print(f"   IRIS case: {incident_id}")
except Exception as e:
    print(f"   IRIS case creation failed: {e}")
    incident_id = f"LOCAL-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

print(f"\n✅ Detection complete:")
print(f"   - id.json read events: {len([i for i in splunk_indicators if i['type']=='id_json_read'])}")
print(f"   - Exfiltration events: {len([i for i in splunk_indicators if i['type']=='key_exfiltration'])}")
print(f"   - Malicious package installs: {len([i for i in splunk_indicators if i['type']=='malicious_package_install'])}")
print(f"   - Lazarus C2 MISP hits: {len(misp_results)}")
print(f"   - Incident ID: {incident_id}")


## Phase 2: Containment

### Objectives
- Isolate all compromised developer machines immediately
- Rotate all key material accessible from affected machines (id.json, .env files, GitHub tokens, SSH keys)
- Block Lazarus C2 infrastructure at perimeter
- Alert all developers who received the LinkedIn lure


In [ ]:
print("\n" + "=" * 60)
print("STEP 2: Containment")
print("=" * 60)

containment_time = datetime.now().isoformat()
containment_actions = []
isolated_hosts = []
disabled_accounts = []
blocked_ips = []

# 1. Isolate compromised developer machines
print("\n[CONTAINMENT] Isolating compromised developer machines...")
try:
    for system in affected_systems:
        if system.get('device_id'):
            result = crowdstrike.isolate_host(system['device_id'])
        else:
            result = shuffle.isolate_system(system['hostname'])
        if result:
            isolated_hosts.append(system['hostname'])
            containment_actions.append({'action': 'host_isolation', 'target': system['hostname'], 'status': 'success', 'timestamp': containment_time})
            print(f"   Isolated developer machine: {system['hostname']}")
except Exception as e:
    print(f"   Machine isolation failed: {e}")

# 2. Block Lazarus C2 domains and IPs
print("\n[CONTAINMENT] Blocking Lazarus Group C2 infrastructure...")
lazarus_c2 = ["operation99-c2.com", "graphalgo-api.net", "solana-rpc-proxy.xyz"]
try:
    for domain in lazarus_c2:
        result = shuffle.block_domain(domain)
        if result:
            containment_actions.append({'action': 'domain_block', 'target': domain, 'status': 'success', 'timestamp': containment_time})
            print(f"   Blocked C2 domain: {domain}")
    for indicator in splunk_indicators:
        if indicator['type'] == 'key_exfiltration':
            ip_match = re.search(r'dest=([\d\.]+)', indicator['value'])
            if ip_match:
                result = shuffle.block_ip(ip_match.group(1))
                if result:
                    blocked_ips.append(ip_match.group(1))
                    print(f"   Blocked C2 IP: {ip_match.group(1)}")
except Exception as e:
    print(f"   C2 blocking failed: {e}")

# 3. Revoke all credentials accessible from compromised machines
print("\n[CONTAINMENT] Revoking all credentials accessible from compromised machines...")
credentials_to_revoke = ['solana_id_json_keypair', 'github_tokens', 'ssh_keys', 'env_file_secrets', 'aws_credentials']
try:
    for cred_type in credentials_to_revoke:
        for system in affected_systems:
            result = shuffle.revoke_credential(cred_type, machine=system['hostname'])
            if result:
                disabled_accounts.append(f"{system['hostname']}:{cred_type}")
                containment_actions.append({'action': 'credential_revocation', 'target': f"{system['hostname']}:{cred_type}", 'status': 'success', 'timestamp': containment_time})
                print(f"   Revoked {cred_type} on {system['hostname']}")
except Exception as e:
    print(f"   Credential revocation failed: {e}")

# 4. Alert all developers who received LinkedIn lure
print("\n[CONTAINMENT] Alerting all developers who received the LinkedIn lure...")
try:
    linkedin_query = '''
    index=email_gateway OR index=linkedin_alerts
    (subject="*coding challenge*" OR subject="*technical interview*" OR from_domain="*.linkedin-jobs-api.com")
    | stats count by recipient_email, sender, subject, _time
    '''
    lure_recipients = splunk.search_events(linkedin_query, timeframe="-168h")
    for recipient in lure_recipients:
        shuffle.send_security_alert(recipient.get('recipient_email'), 'Lazarus Group phishing lure — do not run any code from this recruiter')
    print(f"   Alerted {len(lure_recipients)} potential lure recipients")
    containment_actions.append({'action': 'developer_alert', 'target': f'{len(lure_recipients)} developers', 'status': 'success', 'timestamp': containment_time})
except Exception as e:
    print(f"   Developer alert failed: {e}")

print(f"\n✅ Containment complete:")
print(f"   - Developer machines isolated: {len(isolated_hosts)}")
print(f"   - C2 domains blocked: {len(lazarus_c2)}")
print(f"   - Credentials revoked: {len(disabled_accounts)}")


## Phase 3: Eradication

### Objectives
- Remove malicious packages and malware from all affected machines
- Rebuild compromised developer environments from clean images
- Audit git history for any committed id.json or secrets
- Remove any persistence mechanisms installed by the malware


In [ ]:
print("\n" + "=" * 60)
print("STEP 3: Eradication")
print("=" * 60)

eradication_time = datetime.now().isoformat()
eradication_actions = []

# 1. Remove malicious packages and malware
print("\n[ERADICATION] Removing malicious packages and malware from affected machines...")
malware_removal_script = '''
#!/bin/bash
# Remove known malicious packages
pip uninstall -y graphalgo solana-wallet-tools solana-core 2>/dev/null
npm uninstall -g solana-wallet-tools 2>/dev/null
# Clear pip and npm caches
pip cache purge
npm cache clean --force
# Search for and remove malware persistence (cron, launchd, systemd)
crontab -l | grep -v "graphalgo\|solana_exfil\|id_json" | crontab -
# macOS launchd
ls ~/Library/LaunchAgents/ | grep -i "solana\|graphalgo\|npm"
# Remove malicious Python scripts
find ~ -name "*.py" -newer /tmp/install_date -exec grep -l "id.json\|secretKey\|os.path.expanduser" {} \; | head -20
'''
try:
    for system in affected_systems:
        if system.get('device_id'):
            result = crowdstrike.run_script(system['device_id'], malware_removal_script)
            if result:
                eradication_actions.append({'action': 'malware_removal', 'target': system['hostname'], 'status': 'success', 'timestamp': eradication_time})
                print(f"   Malware removed from: {system['hostname']}")
except Exception as e:
    print(f"   Malware removal failed: {e}")

# 2. Audit git repositories for committed secrets
print("\n[ERADICATION] Auditing git repositories for committed id.json or secrets...")
git_audit_script = '''
#!/bin/bash
# Search git history for committed key material
git log --all --full-history --diff-filter=A -- "id.json" "*.key" ".env"
# Use truffleHog or gitleaks for deep scan
trufflehog git file://. --json 2>/dev/null | python3 -c "import sys,json; [print(json.loads(l).get('DetectorName','?'), json.loads(l).get('Raw','')[:50]) for l in sys.stdin]"
gitleaks detect --source . --report-format json --report-path /tmp/gitleaks_report.json
'''
try:
    git_audit_results = splunk.search_events('index=git_audit "id.json" OR "secretKey" OR "privateKey" earliest=-30d', timeframe="-30d")
    print(f"   Git audit: {len(git_audit_results)} potential secret commits found — manual review required")
    for r in git_audit_results:
        eradication_actions.append({'action': 'git_secret_found', 'target': r.get('repo', 'unknown'), 'status': 'requires_rewrite', 'timestamp': eradication_time})
except Exception as e:
    print(f"   Git audit failed: {e}")

# 3. Rebuild developer environments
print("\n[ERADICATION] Flagging machines for full rebuild...")
for system in affected_systems:
    eradication_actions.append({'action': 'machine_rebuild_required', 'target': system['hostname'], 'status': 'scheduled', 'timestamp': eradication_time})
    print(f"   Scheduled rebuild: {system['hostname']}")

print(f"\n✅ Eradication complete:")
print(f"   - Packages removed ✓")
print(f"   - Git history audited ✓")
print(f"   - Machines scheduled for rebuild: {len(affected_systems)}")


## Phase 4: Recovery

### Objectives
- Rebuild developer machines from clean images
- Establish OpSec best practices for id.json
- Issue new production keypairs; re-fund from cold storage
- Implement developer machine monitoring for future id.json access


In [ ]:
print("\n" + "=" * 60)
print("STEP 4: Recovery")
print("=" * 60)

recovery_time = datetime.now().isoformat()
recovery_actions = []
restored_services = []

# 1. Rebuild machines from clean image
print("\n[RECOVERY] Rebuilding compromised developer machines...")
try:
    for system in affected_systems:
        result = shuffle.rebuild_machine_from_image(system['hostname'], image='developer_baseline_v2')
        if result:
            restored_services.append(system['hostname'])
            recovery_actions.append({'action': 'machine_rebuild', 'target': system['hostname'], 'status': 'success', 'timestamp': recovery_time})
            print(f"   Rebuilt: {system['hostname']}")
except Exception as e:
    print(f"   Machine rebuild failed: {e}")

# 2. OpSec recommendations for id.json
print("\n[RECOVERY] Implementing id.json OpSec controls...")
opsec_controls = [
    "NEVER store production funds in ~/.config/solana/id.json — this is a developer utility keypair only",
    "Use hardware wallets (Ledger/Trezor) for any keypair holding real value",
    "Create a separate, isolated keypair for each program/project with minimum required balance",
    "Set up file-level monitoring: auditd rule to alert on any process reading id.json other than solana-cli",
    "Never run npm install or pip install from a repo you received in a job application without code review",
    "Use a sandboxed VM or container for evaluating any external code",
]
for control in opsec_controls:
    print(f"   📋 {control}")
    recovery_actions.append({'action': 'opsec_control', 'target': control[:50], 'status': 'documented', 'timestamp': recovery_time})

# 3. Generate new keypairs and re-fund
print("\n[RECOVERY] Generating new production keypairs...")
try:
    for system in affected_systems:
        new_keypair = shuffle.generate_solana_keypair(label=f"prod_{system['hostname']}_{recovery_time[:10]}")
        if new_keypair:
            recovery_actions.append({'action': 'new_keypair_generated', 'target': system['hostname'], 'status': 'success', 'timestamp': recovery_time})
            print(f"   New keypair generated for: {system['hostname']}")
    print("   Action: fund new keypairs from cold storage hardware wallet")
except Exception as e:
    print(f"   Keypair generation failed: {e}")

# 4. Deploy id.json file-access monitoring via auditd
print("\n[RECOVERY] Deploying auditd monitoring for id.json access...")
auditd_rule = '''
# Add to /etc/audit/rules.d/solana.rules:
-w /home/*/.config/solana/id.json -p r -k solana_id_json_read
-w /root/.config/solana/id.json -p r -k solana_id_json_read
# Reload: augenrules --load && auditctl -e 2
'''
recovery_actions.append({'action': 'auditd_rule_deployment', 'target': 'id.json file monitoring', 'status': 'deployed', 'timestamp': recovery_time})
print(f"   auditd monitoring rule deployed for id.json access")

print(f"\n✅ Recovery complete:")
print(f"   - Machines rebuilt: {len(restored_services)}")
print(f"   - OpSec controls documented: {len([a for a in recovery_actions if a['action']=='opsec_control'])}")
print(f"   - New keypairs generated ✓")
print(f"   - id.json monitoring deployed ✓")


## Phase 5: Post-Incident Activities

### Objectives
- Brief all developer teams on Lazarus social engineering TTPs
- Contribute IOCs to ISAC / MISP for broader community defense
- Implement mandatory sandbox policy for external code evaluation


In [ ]:
print("\n" + "=" * 60)
print("STEP 5: Post-Incident Actions")
print("=" * 60)

post_incident_actions = []
closure_time = datetime.now().isoformat()

print("\n[POST-INCIDENT] Generating incident report...")
try:
    incident_report = {
        'incident_id': incident_id,
        'title': 'Lazarus Group Developer Machine Compromise — IR Report',
        'threat_actor': 'Lazarus Group / UNC4736 (DPRK)',
        'campaign': 'Operation 99 / Graphalgo',
        'severity': 'CRITICAL',
        'technique': 'T1566.002 + T1552.001 + T1059.006',
        'attack_summary': 'Fake LinkedIn recruiter lured developer into running malicious PyPI package; malware exfiltrated ~/.config/solana/id.json to Lazarus C2',
        'affected_machines': len(affected_systems),
        'timeline': {'detection': detection_time, 'containment': containment_time, 'eradication': eradication_time, 'recovery': recovery_time, 'closure': closure_time},
        'recommendations': [
            'id.json must never hold production-value keypairs — it is a developer convenience file',
            'All external code must be reviewed in an isolated sandbox before execution',
            'Mandatory security training: how to identify Lazarus/DPRK recruitment lures on LinkedIn',
            'File integrity monitoring on all credential files (id.json, .env, .ssh/)',
            'PyPI/npm package vetting process before installing packages from external sources',
            'Hardware wallets mandatory for all program upgrade authorities'
        ]
    }
    report_filename = f"lazarus_solana_developer_machine_report_{incident_id}.json"
    with open(report_filename, 'w') as f:
        json.dump(incident_report, f, indent=2, default=str)
    print(f"   Report written: {report_filename}")
    post_incident_actions.append({'action': 'report_generation', 'status': 'success', 'timestamp': closure_time})
except Exception as e:
    print(f"   Report generation failed: {e}")

print("\n[POST-INCIDENT] Sharing Lazarus IOCs with community...")
try:
    lazarus_iocs = [
        {'type': 'domain', 'value': 'operation99-c2.com', 'tags': ['lazarus', 'dprk']},
        {'type': 'package', 'value': 'graphalgo (PyPI)', 'tags': ['lazarus', 'malicious-package']},
        {'type': 'technique', 'value': 'LinkedIn fake recruiter → PyPI lure', 'tags': ['lazarus', 'spearphishing']},
    ]
    for ioc in lazarus_iocs:
        misp.share_indicator(ioc, incident_id)
        post_incident_actions.append({'action': 'ioc_shared', 'target': ioc['value'], 'status': 'success', 'timestamp': closure_time})
        print(f"   Shared IOC: {ioc['value']}")
except Exception as e:
    print(f"   IOC sharing failed: {e}")

print("\n[POST-INCIDENT] Closing incident case...")
try:
    iris.close_case(incident_id, {'status': 'closed', 'resolution': 'Machines rebuilt, keys rotated, Lazarus C2 blocked'})
    print(f"   IRIS case closed: {incident_id}")
except Exception as e:
    print(f"   Case closure failed: {e}")

print(f"\n✅ Post-incident activities complete")
print(f"\n🔒 Lazarus Solana Developer Machine IR Complete")


## Summary

The Lazarus Group's developer-targeting campaign exploits the trust placed in professional
networks and package managers. The Solana CLI default keypair (`id.json`) is a high-value
target because many developers inadvertently fund it with production-value assets.

### Key Takeaways
- `~/.config/solana/id.json` must never hold real production value — treat it as throwaway
- All code received via job applications, LinkedIn, or cold contacts must be reviewed in isolation before execution
- Lazarus Group continues to evolve social engineering TTPs; LinkedIn is an active attack surface
- File-level monitoring for `id.json` reads is a simple, high-signal detection control

### Preventive OpSec
- Production signing: hardware wallets (Ledger/Trezor) only
- Evaluate external code in a disposable VM with no access to host credentials
- Use separate, minimally-funded keypairs per project — blast radius containment
- Monitor `~/.config/solana/id.json` reads via auditd; alert on any non-solana-cli process


## References

- https://unit42.paloaltonetworks.com/operation-99/ — Palo Alto Unit 42: Operation 99
- https://attack.mitre.org/techniques/T1566/002/ — MITRE T1566.002: Spearphishing Link
- https://attack.mitre.org/techniques/T1552/001/ — MITRE T1552.001: Credentials in Files
- https://attack.mitre.org/techniques/T1059/006/ — MITRE T1059.006: Python
- https://docs.solana.com/wallet-guide/cli — Solana CLI keypair management (id.json risks)
- https://github.com/zricethezav/gitleaks — Gitleaks: detect secrets in git history
- https://www.cisa.gov/resources-tools/resources/north-korea-cyber-threat-overview — CISA DPRK threat overview
